In [ ]:
%%capture
import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from datetime import datetime
from django_pandas.io import read_frame
from pathlib import Path


from dj_notebook import activate

# pd.options.mode.copy_on_write = True
# pd.options.mode.chained_assignment = "raise"
env_file = os.environ["INTECOMM_ENV"]
documents_folder = os.environ["INTECOMM_DOCUMENTS_FOLDER"]
plus = activate(dotenv_file=env_file)

report_folder = Path(documents_folder)
# output is suppressed ut normally would spew out all the edc loading messages


In [ ]:
from django.contrib.sites.models import Site
from django.apps import apps as django_apps
from edc_sites.site import sites
from edc_constants.constants import YES
from edc_pdutils.dataframes import get_crf, get_subject_consent, get_subject_visit

from intecomm_screening.models import PatientLog


In [ ]:
qs = PatientLog.objects.filter(subject_identifier__isnull=False)
df_orig = read_frame(qs)
df_patient_log = df_orig.copy()


In [ ]:
df_patient_log["gender"].value_counts()

In [ ]:
# sites = {obj.domain: obj.id for obj in Site.objects.all()}
sites_data = {s.name.title(): s.site_id for s in sites.all(aslist=True)}
df_patient_log["site"] = df_patient_log["site"].map(sites_data)
df_patient_log["site"] = df_patient_log["site"].apply(pd.to_numeric)
df_patient_log["site"].value_counts()

In [ ]:
df_patient_log["country"] = df_patient_log["site"].apply(lambda x: "ug" if x < 200 else "tz")

In [ ]:
df_patient_log["country"].value_counts()

In [ ]:
data = [
    "Number of clinically stable patients according to routine clinical services (patient log file)", 
    df_patient_log["country"].value_counts()["ug"],
    df_patient_log["country"].value_counts()["tz"], []]
rowdf = pd.DataFrame([data], columns=summary_columns)
rowdf["total"] = rowdf["uganda"] + rowdf["tanzania"]

In [ ]:
summary_columns=["label", "uganda", "tanzania", "total"]
df_summary = pd.DataFrame(columns=summary_columns)
df_summary = pd.concat([df_summary, rowdf])


In [ ]:
df_summary

In [ ]:
import pandas as pd
from edc_analytics.constants import N_ONLY, N_WITH_COL_PROP, N_WITH_ROW_PROP
from edc_analytics.row import RowDefinition, RowDefinitions
from edc_analytics.table import Table


class ScreeningTable(Table):

    countries = {"ug": "Uganda", "tz": "Tanzania"}
    
    def __init__(self, main_df: pd.DataFrame = None):
        super().__init__(colname="", main_df=main_df, title="Screening")

    @property
    def row_definitions(self) -> RowDefinitions:
        df_tmp = self.main_df.copy()
        
        row_defs = RowDefinitions(reverse_rows=False)
        row0 = RowDefinition(
            title="Stable",
            label=self.default_sublabel,
            condition=(df_tmp["gender"].notna()),
            columns={
                "F": (N_ONLY, 2),
                "M": (N_ONLY, 2),
                "All": (N_ONLY, 2),
            },
            drop=False,
        )
        row_defs.add(row0)

        for country_code, country in self.countries.items():
            columns = {
                "F": (N_WITH_COL_PROP, 2),
                "M": (N_WITH_COL_PROP, 2),
                "All": (N_WITH_ROW_PROP, 2),
            }
            row_defs.add(
                RowDefinition(
                    title="Screened" if country_code=="ug" else "",
                    label=country,
                    condition=((df_tmp["country"] == country_code) & (df_tmp["screening_identifier"].notna())),
                    columns=columns,
                    drop=False,
                )
            )

        
        for country_code, country in self.countries.items():
            columns = {
                "F": (N_WITH_COL_PROP, 2),
                "M": (N_WITH_COL_PROP, 2),
                "All": (N_WITH_ROW_PROP, 2),
            }
            row_defs.add(
                RowDefinition(
                    title="Consented" if country_code=="ug" else "",
                    label=country,
                    condition=((df_tmp["country"] == country_code) & (df_tmp["subject_identifier"].str.startswith("107"))),
                    columns=columns,
                    drop=False,
                )
            )

        for country_code, country in self.countries.items():
            columns = {
                "F": (N_WITH_COL_PROP, 2),
                "M": (N_WITH_COL_PROP, 2),
                "All": (N_WITH_ROW_PROP, 2),
            }
            row_defs.add(
                RowDefinition(
                    title="Grouped" if country_code=="ug" else "",
                    label=country,
                    condition=((df_tmp["country"] == country_code) & (df_tmp["group_identifier"].notna())),
                    columns=columns,
                    drop=False,
                )
            )

        for country_code, country in self.countries.items():
            columns = {
                "F": (N_WITH_COL_PROP, 2),
                "M": (N_WITH_COL_PROP, 2),
                "All": (N_WITH_ROW_PROP, 2),
            }
            row_defs.add(
                RowDefinition(
                    title="Groups" if country_code=="ug" else "",
                    label=country,
                    condition=((df_tmp["country"] == country_code) & (df_tmp["age_in_years"])),
                    columns=columns,
                    drop=False,
                )
            )


        return row_defs


In [ ]:
tbl = ScreeningTable(main_df=df)

In [ ]:
tbl.formatted_df


In [ ]:
df.columns

In [ ]:
df_vitals = get_crf("intecomm_subject.vitals", subject_visit_model="intecomm_subject.subjectvisit")

In [ ]:
dfdx = get_crf("intecomm_reports.diagnoses")
dfdx = dfdx[["subject_identifier", "hiv","htn","dm","hiv_dx_date","htn_dx_date","dm_dx_date"]]
df1 = pd.merge(dfdx, df_vitals, on="subject_identifier", how="outer", indicator=False)

df1000 = get_subject_visit("intecomm_subject.subjectvisit")
df1000 = df1000[df1000.visit_code<1001.0]
df1 = pd.merge(df1000, df1, on="subject_identifier", how="outer", indicator=True)

df1["_merge"].value_counts()

In [ ]:
df1["_merge"].value_counts()

In [ ]:

df1[df1["_merge"]!="both"]

In [ ]:
# get randomization table (of group identifiers)

sites_data = {s.name.lower(): s.site_id for s in sites.all(aslist=True)}

allocation_data = {1:"community", 2:"facility"}

qs = django_apps.get_model("intecomm_rando.randomizationlist").objects.all()
df_rando = read_frame(qs)

df_rando["allocation"] = df_rando["allocation"].apply(pd.to_numeric)
df_rando["site"] = df_rando["site_name"].map(sites_data)
df_rando["site"] = df_rando["site"].apply(pd.to_numeric)
df_rando["country"] = df_rando["site"].apply(lambda x: "ug" if x < 200 else "tz")
df_rando["assignment"] = df_rando["allocation"].map({1:"community", 2:"facility"})

df_rando = df_rando.sort_values(["site", "group_identifier"])
df_rando = df_rando[(df_rando["group_identifier"].notna())][["group_identifier", "site", "assignment", "country", "sid"]]
df_rando = df_rando.reset_index(drop=True)


In [ ]:
# assignments are balanced#
df_rando["assignment"].value_counts()


In [ ]:
# number of groups
df_counts = df_rando.groupby(by=["site", "assignment"])[["site"]].count()
df_counts["site"].sum()

In [ ]:
# number of groups per country
df_rando["country"].value_counts()


In [ ]:
# number of groups per site
df_rando["site"].value_counts()


In [ ]:
# subjects consented and grouped
df_consent = get_subject_consent("intecomm_consent.subjectconsent")
df_consent = df_consent[["subject_identifier", "gender", "dob"]]
df_consent = pd.merge(df_consent, df_patient_log[["subject_identifier", "screening_identifier", "group_identifier"]], on="subject_identifier", how="left")
df_consent = df_consent.reset_index(drop=True)

In [ ]:
# merge with rando
df1 = pd.merge(df_consent, df_rando, on="group_identifier", how="left")


In [ ]:
# consented
df1["gender"].value_counts()

In [ ]:
# consented but not randomized / grouped
df1[df1["group_identifier"].isna()]["gender"].value_counts()

In [ ]:
# consented and randomized / grouped
df1 = df1[df1["group_identifier"].notna()]
df1.reset_index(drop=True)
df1["gender"].value_counts()

In [ ]:
dfdx = get_crf("intecomm_reports.diagnoses")
dfdx = dfdx[["subject_identifier", "hiv","htn","dm","hiv_dx_date","htn_dx_date","dm_dx_date"]]
len(dfdx)

In [ ]:
df_subjects = pd.merge(df1, dfdx, on="subject_identifier", how="left")
len(df_subjects)

In [ ]:
df_subjects["gender"].value_counts()

In [ ]:
# total diagnoses ("hiv_dx_date", "htn_dx_date", "dm_dx_date")
df_count = df_subjects[["subject_identifier", "hiv_dx_date", "htn_dx_date", "dm_dx_date"]].count()
df_count[1:4].sum()

In [ ]:
def row(df_stats, cond=None, label=None, title=None, df_subjects=None):
    columns = ["title", "label", "uganda", "tanzania", "total"]
    if not cond.empty:
        stat = df_subjects[cond]["country"].value_counts().to_frame()
    else:
        stat = df_subjects["country"].value_counts().to_frame()
    stat = stat.reset_index()
    stat = pd.pivot_table(stat, columns=["country"], values=["count"], fill_value=0.0)
    stat = stat.reset_index(drop=True)
    stat = stat.rename_axis([None], axis="columns")
    tz, ug = 0.0, 0.0
    if "tz" not in stat.columns:
        stat["tz"] = 0.0
    if "ug" not in stat.columns:
        stat["ug"] = 0.0
    stat["total"] = stat["tz"] + stat["ug"]
    stat = stat.rename(columns={"ug": "uganda", "tz": "tanzania"})
    stat["title"] = title or ""
    stat["label"] = label
    stat = stat[columns]
    if df_stats.empty:
        return stat.copy()
    return pd.concat([df_stats, stat])


In [ ]:
df_dx_stats = row(pd.DataFrame(), cond=pd.Series(), title="Subjects", label="All", df_subjects=df_subjects)
df_dx_stats

In [ ]:
# hiv only
cond = (df_subjects["hiv"]==True) & (df_subjects["htn"]==False) & (df_subjects["dm"]==False)
df_dx_stats = row(df_dx_stats, cond=cond, label="HIV only", df_subjects=df_subjects)
df_dx_stats


In [ ]:
# htn only
cond = (df_subjects["hiv"]==False) & (df_subjects["htn"]==True) & (df_subjects["dm"]==False)
df_dx_stats = row(df_dx_stats, cond=cond, label="HTN only", df_subjects=df_subjects)
df_dx_stats


In [ ]:
# dm only
cond = (df_subjects["hiv"]==False) & (df_subjects["htn"]==False) & (df_subjects["dm"]==True)
df_dx_stats = row(df_dx_stats, cond=cond, label="DM only", df_subjects=df_subjects)
df_dx_stats


In [ ]:
# hiv + htn
cond = (df_subjects["hiv"]==True) & (df_subjects["htn"]==True) & (df_subjects["dm"]==False)
df_dx_stats = row(df_dx_stats, cond=cond, label="HIV and HTN", df_subjects=df_subjects)
df_dx_stats


In [ ]:
# hiv + dm
cond = (df_subjects["hiv"]==True) & (df_subjects["htn"]==False) & (df_subjects["dm"]==True)
df_dx_stats = row(df_dx_stats, cond=cond, label="HIV and DM", df_subjects=df_subjects)
df_dx_stats


In [ ]:
# htn + dm
cond = (df_subjects["hiv"]==False) & (df_subjects["htn"]==True) & (df_subjects["dm"]==True)
df_dx_stats = row(df_dx_stats, cond=cond, label="HTN and DM", df_subjects=df_subjects)
df_dx_stats


In [ ]:
# not any 
cond = ((df_subjects["hiv"].isna()) & (df_subjects["htn"].isna()) & (df_subjects["dm"].isna()))
df_dx_stats = row(df_dx_stats, cond=cond, label="isna", df_subjects=df_subjects)
df_dx_stats


In [ ]:
df_dx_stats[1:]["total"].sum()

In [ ]:
df_dx_stats[1:]["uganda"].sum()

In [ ]:
df_dx_stats[1:]["tanzania"].sum()